Project 3, Person A: Task 3.1, Task 3.2, and the Part II AI exercise

Code follows the official documentation examples of the tools we use. Every code cell starts with a comment that says either Source, naming the documentation page and example the code follows, or Custom, saying why no example fits and what reference the code is based on. The companion document, Project 3 Person A Reference, links each decision to its documentation example.

To run it, data/Online_Retail.xlsx and data/groceries.csv must sit in a data folder beside this notebook.

In [ ]:
%pip install mlxtend openpyxl

In [ ]:
import csv
import tracemalloc

import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

Task 3.1: Data Preprocessing

In [ ]:
# loading retail data 
retail = pd.read_excel("data/Online_Retail.xlsx", dtype={"InvoiceNo": str, "StockCode": str})
print(retail.shape)

In [ ]:
# remove missing item ID's 
retail = retail.dropna(subset=["StockCode", "Description"])                  

# cleaning other items 
    # cancellations and adjustments 
    # returns 
    # stock notes 
    # non product codes 
retail = retail[~retail["InvoiceNo"].str.startswith(("C", "c", "A"))          
    & (retail["Quantity"] > 0) 
    & (retail["UnitPrice"] > 0)         
    & retail["StockCode"].str.fullmatch(r"\d{5}[A-Za-z]*")]       

# combines stock codes with mixed cases 
retail["StockCode"] = retail["StockCode"].str.upper()                          
print(retail.shape, "| invoices:", retail["InvoiceNo"].nunique(), "| products:", retail["StockCode"].nunique())

In [ ]:
# panda user guide for aggregation 
retail_baskets = retail.groupby("InvoiceNo")["StockCode"].agg(lambda x: set(x)).tolist()

# loading grocercy data 
with open("data/groceries.csv", newline="") as csvfile:
    groceries_baskets = list(csv.reader(csvfile))

print("Online Retail baskets:", len(retail_baskets), "| Groceries baskets:", len(groceries_baskets))

In [ ]:
# mlxtend datafram examples  
te = TransactionEncoder()
te_ary = te.fit(retail_baskets).transform(retail_baskets)
retail_df = pd.DataFrame(te_ary, columns=te.columns_)

te = TransactionEncoder()
te_ary = te.fit(groceries_baskets).transform(groceries_baskets)
groceries_df = pd.DataFrame(te_ary, columns=te.columns_)

print("Online Retail:", retail_df.shape, "| Groceries:", groceries_df.shape)

Task 3.2: Apriori vs. FP-Growth

In [ ]:
# Source: mlxtend fpgrowth user guide, Example 2 (Apriori versus FPGrowth): these are its three calls
SETTINGS = {
    "apriori": lambda df, ms: apriori(df, min_support=ms),
    "apriori low_memory": lambda df, ms: apriori(df, min_support=ms, low_memory=True),
    "fpgrowth": lambda df, ms: fpgrowth(df, min_support=ms),
}
DATASETS = {"Groceries": groceries_df, "Online Retail": retail_df}

# specs range
MINSUP_GRID = [0.05, 0.02, 0.01, 0.005, 0.002, 0.001]

# needed do to failure on running out of memory on laptop
NOT_RUN = {("Online Retail", "apriori", ms) for ms in [0.01, 0.005, 0.002, 0.001]}

In [ ]:
results = []
for dataset, df in DATASETS.items():
    for setting, run in SETTINGS.items():
        for ms in MINSUP_GRID:
            if (dataset, setting, ms) in NOT_RUN:
                results.append({"dataset": dataset, "setting": setting, "min_support": ms, "status": "not run (memory)"})
                continue

            # time runs 
            timing = %timeit -o -q -n 1 -r 3 run(df, ms)

            # spec req for peak mem usage 
            tracemalloc.start()
            run(df, ms)
            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            results.append({"dataset": dataset, "setting": setting, "min_support": ms, "status": "ok",
                            "runtime_s": timing.best, "peak_mb": peak / 1e6})
            print(dataset, "|", setting, "|", ms, "| done")

pd.DataFrame(results)

Part II: AI Integration Protocol and Prompt Logging

prompt: generate code for candidate rule generation using Apriori

    the AI skipped rule generation no support counting, rules or confidence

In [ ]:
from itertools import combinations

def apriori_gen(L_prev):
    candidates = set()

    # Change 1: did nothing, because sorted() compares frozensets by subset, not by their items
    L_list = sorted(L_prev, key=sorted)

    for i in range(len(L_list)):
        l1 = sorted(L_list[i])
        for j in range(i + 1, len(L_list)):
            l2 = sorted(L_list[j])

            # Change 2: once the first k-2 items stop matching, no later itemset can match
            if l1[:-1] != l2[:-1]:
                break

            candidate = L_list[i] | L_list[j]

            # Change 3: check subsets against the original set L_prev, instant lookup vs item by item
            if all(frozenset(s) in L_prev for s in combinations(candidate, len(candidate) - 1)):
                candidates.add(candidate)

    return candidates

L2 = {frozenset(p) for p in [("A","B"), ("A","C"), ("A","D"), ("B","C"), ("B","D"), ("C","D")]}
for c in sorted(apriori_gen(L2), key=sorted):
       print(sorted(c))